## Libraries

In [1]:
# Basic libraries for data science
import pandas as pd

# Classic ML libraries
from sklearn.preprocessing import LabelEncoder
from sklearn.model_selection import train_test_split
from sklearn.metrics import (
    roc_auc_score,
    accuracy_score,
    f1_score,
    average_precision_score,
)
import lightgbm as lgb
import optuna

# Libraries for experiment tracker
import wandb
from optuna.integration.wandb import WeightsAndBiasesCallback

/Users/conquerormikrokosmos/Downloads/LAPTOP MAC/MYUNIVERSITY/ĐẠI HỌC QUỐC GIA TPHCM/ĐH KHOA HỌC TỰ NHIÊN/NĂM 4/HKI/Intelligent Data Analysis/Chatbot_Extension/.venv/lib/python3.10/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


## Load and Prepare Data

In [2]:
# Load preprocessed datasets
train_df = pd.read_csv("data/preprocessed/train_cleaned.csv", index_col=False)
test_df = pd.read_csv("data/preprocessed/test_cleaned.csv", index_col=False)

In [3]:
train_df.head()

,id,Name,Gender,Age,City,Profession,Academic Pressure,Work Pressure,CGPA,Study Satisfaction,Job Satisfaction,Dietary Habits,Degree,Have you ever had suicidal thoughts ?,Work/Study Hours,Financial Stress,Family History of Mental Illness,Depression,SleepDuration_num,is_student
0,0,Aaradhya,Female,49.0,Ludhiana,Chef,0.0,5.0,0.00,0.0,2.0,Healthy,BHM,No,1.0,2.0,No,0,7.5,0
1,1,Vivan,Male,26.0,Varanasi,Teacher,0.0,4.0,0.00,0.0,3.0,Unhealthy,LLB,Yes,7.0,3.0,No,1,4.5,0
2,2,Yuvraj,Male,33.0,Visakhapatnam,Teacher,5.0,0.0,8.97,2.0,0.0,Healthy,B.Pharm,Yes,3.0,1.0,No,1,5.5,1
3,3,Yuvraj,Male,22.0,Mumbai,Teacher,0.0,5.0,0.00,0.0,1.0,Moderate,BBA,Yes,10.0,1.0,Yes,1,4.5,0
4,4,Rhea,Female,30.0,Kanpur,Business Analyst,0.0,1.0,0.00,0.0,1.0,Unhealthy,BBA,Yes,9.0,4.0,Yes,0,5.5,0


In [4]:
test_df.head()

,id,Name,Gender,Age,City,Profession,Academic Pressure,Work Pressure,CGPA,Study Satisfaction,Job Satisfaction,Dietary Habits,Degree,Have you ever had suicidal thoughts ?,Work/Study Hours,Financial Stress,Family History of Mental Illness,SleepDuration_num,is_student
0,140700,Shivam,Male,53.0,Visakhapatnam,Judge,0.0,2.0,0.00,0.0,5.0,Moderate,LLB,No,9.0,3.0,Yes,4.5,0
1,140701,Sanya,Female,58.0,Kolkata,Educational Consultant,0.0,2.0,0.00,0.0,4.0,Moderate,B.Ed,No,6.0,4.0,No,4.5,0
2,140702,Yash,Male,53.0,Jaipur,Teacher,0.0,4.0,0.00,0.0,1.0,Moderate,B.Arch,Yes,12.0,4.0,No,7.5,0
3,140703,Nalini,Female,23.0,Rajkot,Teacher,5.0,0.0,6.84,1.0,0.0,Moderate,BSc,Yes,10.0,4.0,No,7.5,1
4,140704,Shaurya,Male,47.0,Kalyan,Teacher,0.0,5.0,0.00,0.0,5.0,Moderate,BCA,Yes,3.0,4.0,No,7.5,0


In [5]:
# Prepare data for training
drop_cols = ["id", "Name"]
X = train_df.drop(columns=drop_cols + ["Depression"])
y = train_df["Depression"]
X_test_submission = test_df.drop(
    columns=drop_cols
)  # Not have id column, need to concat back after train and inference

In [6]:
# Identify categorical columns (object type)
cat_cols = X.select_dtypes(include=["object"]).columns.tolist()
cat_cols

['Gender',
 'City',
 'Profession',
 'Dietary Habits',
 'Degree',
 'Have you ever had suicidal thoughts ?',
 'Family History of Mental Illness']

In [7]:
# Encode categorical variables
combined = pd.concat([X, X_test_submission], axis=0)

for col in cat_cols:
    le = LabelEncoder()
    # Convert to string to handle potential mixed types
    combined[col] = le.fit_transform(combined[col].astype(str))

combined.head()

,Gender,Age,City,Profession,Academic Pressure,Work Pressure,CGPA,Study Satisfaction,Job Satisfaction,Dietary Habits,Degree,Have you ever had suicidal thoughts ?,Work/Study Hours,Financial Stress,Family History of Mental Illness,SleepDuration_num,is_student
0,0,49.0,62,13,0.0,5.0,0.00,0.0,2.0,11,50,0,1.0,2.0,0,7.5,0
1,1,26.0,118,71,0.0,4.0,0.00,0.0,3.0,32,92,1,7.0,3.0,0,4.5,0
2,1,33.0,123,71,5.0,0.0,8.97,2.0,0.0,11,34,1,3.0,1.0,0,5.5,1
3,1,22.0,79,71,0.0,5.0,0.00,0.0,1.0,22,44,1,10.0,1.0,1,4.5,0
4,0,30.0,46,12,0.0,1.0,0.00,0.0,1.0,32,44,1,9.0,4.0,1,5.5,0


In [8]:
# Split back into train and test
X = combined.iloc[: len(X)]
X_test_submission = combined.iloc[len(X) :]

In [9]:
# Split training data for validation (80% train, 20% validation)
X_train, X_valid, y_train, y_valid = train_test_split(
    X, y, test_size=0.2, random_state=607, stratify=y
)

In [10]:
X_train.head()

,Gender,Age,City,Profession,Academic Pressure,Work Pressure,CGPA,Study Satisfaction,Job Satisfaction,Dietary Habits,Degree,Have you ever had suicidal thoughts ?,Work/Study Hours,Financial Stress,Family History of Mental Illness,SleepDuration_num,is_student
46124,0,47.0,88,48,0.0,5.0,0.0,0.0,2.0,11,30,0,2.0,4.0,1,7.5,0
95387,1,41.0,26,30,0.0,1.0,0.0,0.0,4.0,32,138,1,12.0,2.0,1,4.5,0
123153,1,57.0,88,37,0.0,3.0,0.0,0.0,4.0,32,92,0,2.0,4.0,0,4.5,0
113283,0,56.0,14,71,0.0,2.0,0.0,0.0,5.0,11,31,1,3.0,2.0,0,7.5,0
49710,0,57.0,53,71,0.0,3.0,0.0,0.0,1.0,11,30,0,4.0,1.0,1,5.5,0


In [14]:
y_train.head()

46124     0
95387     0
123153    0
113283    0
49710     0
Name: Depression, dtype: int64

## Model Training

- Trong các bài toán lâm sàng như thế này, thường thì vô tình phát hiện nhầm còn hơn vô tình phát hiện thiếu. Do đó, false negative thường nguy hiểm hơn là một vài false positive (false alarm).

In [12]:
lgb_params_basic = {
    "objective": "binary",
    "metric": "auc",
    "num_leaves": 31,  # num_leaves < 2^(max_depth)
    "max_depth": 5,
    "learning_rate": 0.05,
    "n_estimators": 500,
    "num_threads": 6,  # use an appropriate amount of CPU cores
    "random_state": 607,
    "is_unbalance": True,
}

model_basic = lgb.LGBMClassifier(**lgb_params_basic)
print(model_basic)

LGBMClassifier(is_unbalance=True, learning_rate=0.05, max_depth=5, metric='auc',
               n_estimators=500, num_threads=6, objective='binary',
               random_state=607)


In [13]:
model_basic.fit(
    X_train,
    y_train,
    eval_set=[(X_valid, y_valid)],
    eval_metric="auc",
)

[LightGBM] [Warning] Found whitespace in feature_names, replace with underlines
[LightGBM] [Info] Number of positive: 20454, number of negative: 92106
[LightGBM] [Info] Auto-choosing row-wise multi-threading, the overhead of testing was 0.002427 seconds.
You can set `force_row_wise=true` to remove the overhead.
And if memory is not enough, you can set `force_col_wise=true`.
[LightGBM] [Info] Total Bins 492
[LightGBM] [Info] Number of data points in the train set: 112560, number of used features: 17
[LightGBM] [Warning] Found whitespace in feature_names, replace with underlines
[LightGBM] [Info] [binary:BoostFromScore]: pavg=0.181716 -> initscore=-1.504762
[LightGBM] [Info] Start training from score -1.504762
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain

,boosting_type,'gbdt'
,num_leaves,31
,max_depth,5
,learning_rate,0.05
,n_estimators,500
,subsample_for_bin,200000
,objective,'binary'
,class_weight,None
,min_split_gain,0.0
,min_child_weight,0.001
,min_child_samples,20


In [14]:
y_pred_proba = model_basic.predict_proba(X_valid)[:, 1]
print(type(y_pred_proba))
y_pred_label = (y_pred_proba >= 0.5).astype(int)

print("Valid AUC:", roc_auc_score(y_valid, y_pred_proba))
print("Valid accuracy:", accuracy_score(y_valid, y_pred_label))

<class 'numpy.ndarray'>
Valid AUC: 0.9741717668807587
Valid accuracy: 0.9182302771855011


In [41]:
wandb_kwargs = {
    "entity": "team-csc17001-ida",
    "project": "depression-detection",
    "name": "optuna_lgbm_study5",
}

wandb_callback = WeightsAndBiasesCallback(
    metric_name="valid_ap",  # how the metric will be called in W&B
    wandb_kwargs=wandb_kwargs,  # passed to wandb.init(...)
    as_multirun=False,  # one W&B run for entire study
)

/var/folders/yf/bvb_cdgn46s4l8cp8l3wd_nc0000gn/T/ipykernel_66383/2884670172.py:7: ExperimentalWarning: WeightsAndBiasesCallback is experimental (supported from v2.9.0). The interface can change in the future.
  wandb_callback = WeightsAndBiasesCallback(


In [42]:
@wandb_callback.track_in_wandb()
def objective(trial: optuna.trial.Trial) -> float:
    """
    Thư viện Optuna yêu cầu người dùng tự định nghĩa các hàm objective như thế này.
    Người dùng thường sẽ muốn hàm objective trả về giá trị lớn nhất hoặc nhỏ nhất cho mô hình của mình.
    """

    # 1. Define search space (bounds inspired by LightGBM docs + common Kaggle practice)
    num_leaves = trial.suggest_int("num_leaves", 16, 256)
    max_depth = trial.suggest_int("max_depth", 3, 12)
    learning_rate = trial.suggest_float("learning_rate", 1e-3, 0.3, log=True)
    min_child_samples = trial.suggest_int("min_child_samples", 5, 100)
    subsample = trial.suggest_float("subsample", 0.5, 1.0)  # bagging_fraction
    colsample_bytree = trial.suggest_float(
        "colsample_bytree", 0.5, 1.0
    )  # feature_fraction
    lambda_l1 = trial.suggest_float("lambda_l1", 1e-8, 10.0, log=True)
    lambda_l2 = trial.suggest_float("lambda_l2", 1e-8, 10.0, log=True)

    # Threshold as a hyperparameter
    # Có thể điều chỉnh khoảng [0.1, 0.9] tùy bài toán / class imbalance
    threshold = trial.suggest_float("threshold", 0.1, 0.9)

    # 2. Construct model params
    params = {
        "objective": "binary",
        "is_unbalance": True,
        "metric": "auc",
        "num_leaves": num_leaves,
        "max_depth": max_depth,  # Người ta khuyên là max_depth nên bé hơn hoặc bằng log_2(num_leaves)
        "learning_rate": learning_rate,
        "min_child_samples": min_child_samples,
        "subsample": subsample,
        "colsample_bytree": colsample_bytree,
        "reg_alpha": lambda_l1,
        "reg_lambda": lambda_l2,
        "n_estimators": 2000,  # big; rely on early stopping
        "num_threads": 6,
        "random_state": 607,
    }

    model = lgb.LGBMClassifier(**params)

    # 3. Train with early stopping to avoid overfitting
    model.fit(
        X_train,
        y_train,
        eval_set=[(X_valid, y_valid)],
        eval_metric="average_precision",
        callbacks=[
            lgb.early_stopping(stopping_rounds=50),
            lgb.log_evaluation(period=0),
        ],
    )

    # 4. Evaluate on validation set
    # NOTE: dùng model chứ không phải model_basic
    y_pred_proba = model.predict_proba(X_valid)[:, 1]

    # Using threshold sampled by Optuna for this trial
    y_hat = (y_pred_proba >= threshold).astype(int)

    # Calculate metrics for this specific trial in the whole study
    # AUC và AP dùng predicted probabilities; F1 dùng thresholded class labels
    auc = roc_auc_score(y_valid, y_pred_proba)  # Diện tích dưới đường cong ROC
    ap = average_precision_score(y_valid, y_pred_proba)  # Diện tích dưới đường cong PR
    f1_macro = f1_score(
        y_valid, y_hat, average="macro"
    )  # Chỉ số F1 tại threshold đang xét

    # Log to Optuna for analysis (bao gồm threshold)
    trial.set_user_attr("valid_auc", auc)
    trial.set_user_attr("valid_ap", ap)
    trial.set_user_attr("valid_f1", f1_macro)
    trial.set_user_attr("threshold", threshold)

    # Log to W&B
    wandb.log(
        {
            "valid_auc": auc,
            "valid_ap": ap,
            "valid_f1_macro": f1_macro,
            "threshold": threshold,
        }
    )

    # Objective cho Optuna
    print("Optimizing Average Precision Score")
    return ap  # Optuna will MAXIMIZE this

/var/folders/yf/bvb_cdgn46s4l8cp8l3wd_nc0000gn/T/ipykernel_66383/122470806.py:1: ExperimentalWarning: optuna_integration.wandb.wandb.WeightsAndBiasesCallback.track_in_wandb is experimental (supported from v3.0.0). The interface can change in the future.
  @wandb_callback.track_in_wandb()


In [43]:
study = optuna.create_study(direction="maximize")

[I 2025-11-27 15:06:03,553] A new study created in memory with name: no-name-779fce89-afed-42f5-9663-f175fdf7189c


In [ ]:
study.optimize(
    objective,
    n_trials=100,
    callbacks=[wandb_callback],
    n_jobs=1,
)
wandb.finish()

In [45]:
# Get the best hyper params from study object
best_params = study.best_trial.params
best_params

{'num_leaves': 240,
 'max_depth': 4,
 'learning_rate': 0.09995896705073841,
 'min_child_samples': 55,
 'subsample': 0.6511856126273549,
 'colsample_bytree': 0.5360644669787153,
 'lambda_l1': 0.00014623461031687177,
 'lambda_l2': 5.773602684816733,
 'threshold': 0.5107309048999379}

In [46]:
raise Exception("Stop here to avoid running inference unintentionally")

Exception: Stop here to avoid running inference unintentionally

In [11]:
best_params = {
    "num_leaves": 240,
    "max_depth": 4,
    "learning_rate": 0.09995896705073841,
    "min_child_samples": 55,
    "subsample": 0.6511856126273549,
    "colsample_bytree": 0.5360644669787153,
    "lambda_l1": 0.00014623461031687177,
    "lambda_l2": 5.773602684816733,
    "threshold": 0.5107309048999379,
}

In [12]:
# Use the best hyper params to train final model
final_model = lgb.LGBMClassifier(**best_params)
print(final_model)

LGBMClassifier(colsample_bytree=0.5360644669787153,
               lambda_l1=0.00014623461031687177, lambda_l2=5.773602684816733,
               learning_rate=0.09995896705073841, max_depth=4,
               min_child_samples=55, num_leaves=240,
               subsample=0.6511856126273549, threshold=0.5107309048999379)


In [13]:
# Train the final model with best hyperparameters on the entire training set
final_model.fit(
    X,
    y,
    eval_set=[(X_valid, y_valid)],
    eval_metric="auc",
)

[LightGBM] [Warning] Unknown parameter: threshold
[LightGBM] [Warning] lambda_l1 is set=0.00014623461031687177, reg_alpha=0.0 will be ignored. Current value: lambda_l1=0.00014623461031687177
[LightGBM] [Warning] lambda_l2 is set=5.773602684816733, reg_lambda=0.0 will be ignored. Current value: lambda_l2=5.773602684816733
[LightGBM] [Warning] Found whitespace in feature_names, replace with underlines
[LightGBM] [Warning] Unknown parameter: threshold
[LightGBM] [Warning] lambda_l1 is set=0.00014623461031687177, reg_alpha=0.0 will be ignored. Current value: lambda_l1=0.00014623461031687177
[LightGBM] [Warning] lambda_l2 is set=5.773602684816733, reg_lambda=0.0 will be ignored. Current value: lambda_l2=5.773602684816733
[LightGBM] [Info] Number of positive: 25567, number of negative: 115133
[LightGBM] [Info] Auto-choosing row-wise multi-threading, the overhead of testing was 0.003185 seconds.
You can set `force_row_wise=true` to remove the overhead.
And if memory is not enough, you can set

,boosting_type,'gbdt'
,num_leaves,240
,max_depth,4
,learning_rate,0.09995896705073841
,n_estimators,100
,subsample_for_bin,200000
,objective,None
,class_weight,None
,min_split_gain,0.0
,min_child_weight,0.001
,min_child_samples,55


In [15]:
# Get validation metrics
y_pred_proba = final_model.predict_proba(X_valid)[:, 1]
# Get the best threshold from the study
best_threshold = 0.5107309048999379
y_pred_label = (y_pred_proba >= best_threshold).astype(int)
print("Final Valid AUC:", roc_auc_score(y_valid, y_pred_proba))
print("Final Valid accuracy:", accuracy_score(y_valid, y_pred_label))
print("Final Valid F1 Macro:", f1_score(y_valid, y_pred_label, average="macro"))
print("Final Valid AP:", average_precision_score(y_valid, y_pred_proba))

[LightGBM] [Warning] Unknown parameter: threshold
[LightGBM] [Warning] lambda_l1 is set=0.00014623461031687177, reg_alpha=0.0 will be ignored. Current value: lambda_l1=0.00014623461031687177
[LightGBM] [Warning] lambda_l2 is set=5.773602684816733, reg_lambda=0.0 will be ignored. Current value: lambda_l2=5.773602684816733
Final Valid AUC: 0.9753527375167568
Final Valid accuracy: 0.939729921819474
Final Valid F1 Macro: 0.8964867315454017
Final Valid AP: 0.9075326920639003


In [16]:
# Get confusion matrix
from sklearn.metrics import confusion_matrix, classification_report

cm = confusion_matrix(y_valid, y_pred_label)
print("Confusion Matrix:\n", cm)

Confusion Matrix:
 [[22316   711]
 [  985  4128]]


In [17]:
# Get classifiaction report
cr = classification_report(y_valid, y_pred_label)
print("Classification Report:\n", cr)

Classification Report:
               precision    recall  f1-score   support

           0       0.96      0.97      0.96     23027
           1       0.85      0.81      0.83      5113

    accuracy                           0.94     28140
   macro avg       0.91      0.89      0.90     28140
weighted avg       0.94      0.94      0.94     28140



In [18]:
# Prepare K-Fold Cross-Validation on the entire training set
from sklearn.model_selection import KFold

kf = KFold(n_splits=5, shuffle=True, random_state=607)

In [19]:
# Run K-Fold CV and get mean roc_auc, accuracy, f1_macro, average_precision
roc_auc_scores = []
accuracy_scores = []
f1_macro_scores = []
average_precision_scores = []

for train_index, valid_index in kf.split(X):
    X_train_kf, X_valid_kf = X.iloc[train_index], X.iloc[valid_index]
    y_train_kf, y_valid_kf = y.iloc[train_index], y.iloc[valid_index]

    model_kf = lgb.LGBMClassifier(**best_params)
    model_kf.fit(
        X_train_kf,
        y_train_kf,
        eval_set=[(X_valid_kf, y_valid_kf)],
        eval_metric="auc",
    )

    y_pred_proba_kf = model_kf.predict_proba(X_valid_kf)[:, 1]
    y_pred_label_kf = (y_pred_proba_kf >= best_threshold).astype(int)

    roc_auc_scores.append(roc_auc_score(y_valid_kf, y_pred_proba_kf))
    accuracy_scores.append(accuracy_score(y_valid_kf, y_pred_label_kf))
    f1_macro_scores.append(f1_score(y_valid_kf, y_pred_label_kf, average="macro"))
    average_precision_scores.append(
        average_precision_score(y_valid_kf, y_pred_proba_kf)
    )

print("K-Fold CV Mean AUC:", sum(roc_auc_scores) / len(roc_auc_scores))
print("K-Fold CV Mean Accuracy:", sum(accuracy_scores) / len(accuracy_scores))
print("K-Fold CV Mean F1 Macro:", sum(f1_macro_scores) / len(f1_macro_scores))
print(
    "K-Fold CV Mean AP:", sum(average_precision_scores) / len(average_precision_scores)
)

[LightGBM] [Warning] Unknown parameter: threshold
[LightGBM] [Warning] lambda_l1 is set=0.00014623461031687177, reg_alpha=0.0 will be ignored. Current value: lambda_l1=0.00014623461031687177
[LightGBM] [Warning] lambda_l2 is set=5.773602684816733, reg_lambda=0.0 will be ignored. Current value: lambda_l2=5.773602684816733
[LightGBM] [Warning] Found whitespace in feature_names, replace with underlines
[LightGBM] [Warning] Unknown parameter: threshold
[LightGBM] [Warning] lambda_l1 is set=0.00014623461031687177, reg_alpha=0.0 will be ignored. Current value: lambda_l1=0.00014623461031687177
[LightGBM] [Warning] lambda_l2 is set=5.773602684816733, reg_lambda=0.0 will be ignored. Current value: lambda_l2=5.773602684816733
[LightGBM] [Info] Number of positive: 20488, number of negative: 92072
[LightGBM] [Info] Auto-choosing row-wise multi-threading, the overhead of testing was 0.002515 seconds.
You can set `force_row_wise=true` to remove the overhead.
And if memory is not enough, you can set 

In [20]:
# Get predicted probabilities on test set
y_test_proba = final_model.predict_proba(X_test_submission)[:, 1]
# Get the best threshold from the study
best_threshold = study.best_trial.user_attrs["threshold"]
y_test_label = (y_test_proba >= best_threshold).astype(int)

[LightGBM] [Warning] Unknown parameter: threshold
[LightGBM] [Warning] lambda_l1 is set=0.00014623461031687177, reg_alpha=0.0 will be ignored. Current value: lambda_l1=0.00014623461031687177
[LightGBM] [Warning] lambda_l2 is set=5.773602684816733, reg_lambda=0.0 will be ignored. Current value: lambda_l2=5.773602684816733


NameError: name 'study' is not defined

In [ ]:
# Prepare submission file
submission_df = pd.read_csv("sample_submission.csv")
submission_df["Depression"] = y_test_label
submission_df.to_csv("lgbm_submission.csv", index=False)